# Apply marine quality control functions on MAROB data

In [1]:
%load_ext autoreload
%autoreload 2

## Import python libraries

In [2]:
import requests

In [3]:
import pandas as pd

In [4]:
from marine_qc import (
    do_position_check, 
    do_date_check, 
    do_time_check, 
    do_missing_value_check,
    do_hard_limit_check, 
    do_sst_freeze_check,
    do_supersaturation_check,
    do_wind_consistency_check,
    do_multiple_individual_check,
    do_track_check,
)

In [5]:
from cdm_reader_mapper import map_model

C:\Users\llierham\mobaxterm\.venvs\mp_py\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load remote data into a pandas DataFrame

The daily data is stored remotely as json dictionaries. The data will be overwritten in the night with the data from the previous day. 

In [6]:
url = "https://oflks472.dwd.de:3443/api/marob_yesterday"

In [7]:
res =  requests.get(url, verify=False)

C:\Users\llierham\mobaxterm\.venvs\mp_py\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'oflks472.dwd.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [8]:
data = res.json()

In [9]:
df = pd.DataFrame(data)
df

,IID,marob_id,kennung,geogr_laenge,geogr_laenge_flag,geogr_breite,geogr_breite_flag,stationshoehe_msl,barometerhoehe_msl,messzeit,...,luftdruck_reduziert,luftdruck_reduziert_flag,wassertemperatur,wassertemperatur_flag,messtiefe,sensorhoehe_was_ff,windrichtung,windrichtung_flag,windgeschwindigkeit,windgeschwindigkeit_flag
0,10384,441917752,0701/,-125.60,NaN,47.60,NaN,NaN,NaN,2026-04-07T01:00:00,...,1020.3,None,NaN,None,NaN,NaN,NaN,None,NaN,None
1,10384,441936176,0703/,-125.70,NaN,47.20,NaN,NaN,NaN,2026-04-07T03:00:00,...,1020.8,None,NaN,None,NaN,NaN,NaN,None,NaN,None
2,10384,441955660,0705/,-125.70,NaN,46.80,NaN,NaN,NaN,2026-04-07T05:00:00,...,1020.7,None,NaN,None,NaN,NaN,NaN,None,NaN,None
3,10384,441973355,0707/,-125.70,NaN,46.30,NaN,NaN,NaN,2026-04-07T07:00:00,...,1020.9,None,NaN,None,NaN,NaN,NaN,None,NaN,None
4,10384,441991207,0709/,-125.70,NaN,45.90,NaN,NaN,NaN,2026-04-07T09:00:00,...,1021.0,None,NaN,None,NaN,NaN,NaN,None,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18166,10384,442063883,ZZB5AKK,-1.43,NaN,-12.49,NaN,NaN,56.0,2026-04-07T19:00:00,...,1014.0,None,NaN,None,NaN,57.0,225.0,None,2.6,None
18167,10384,442072460,ZZB5AKK,-1.26,NaN,-12.72,NaN,NaN,56.0,2026-04-07T20:00:00,...,1014.6,None,NaN,None,NaN,57.0,195.0,None,1.8,None
18168,10384,442081846,ZZB5AKK,-1.09,NaN,-12.95,NaN,NaN,56.0,2026-04-07T21:00:00,...,1015.0,None,NaN,None,NaN,57.0,140.0,None,2.4,None
18169,10384,442090249,ZZB5AKK,-0.93,NaN,-13.18,NaN,NaN,56.0,2026-04-07T22:00:00,...,1015.2,None,NaN,None,NaN,57.0,130.0,None,5.2,None


## Do Quality Control Checks

We use `marine_qc` to do some quality control checks on each individual report:

* We start with some positional check.
* Then we do a datime check existing of a date and a time check
* Afterwards we do some checks on the observed sea surface temperature. This is a hard limit check, a missing value check and a freeze check
* These checks will be performed at once using a quality control dictionary containing all relevant inforamtion.
* The the end we do some check on two observed variables. This is a supersaturation and a wind consistency check.

The next step is to do some sequential checks:

* The first check is a simple track check
* We end with a spike check of some observational values

The last part of the quality control are some buddy check. We compare observational values with other values within a spatial and a temporal range:

* We start with a MDS buddy cehck
* and close with a bayesian buddy check

### Do Quality Control checks on individual reports

#### do_positional_check

We start with a postition check. This check tests whether latitude and longitude values are within valid ranges.

-90 <= latitude <= 90

-180 <= longitude <= 180

In [10]:
pos_qc = do_position_check(
    lat=df["geogr_breite"],
    lon=df["geogr_laenge"],
)

Now we plot the lines corresponding to failed positions (the flag for failed is `1`).

In [11]:
df[pos_qc == 1][["geogr_laenge", "geogr_breite"]]

,geogr_laenge,geogr_breite
4287,439.72,17.3
6050,459.72,22.1
6067,449.72,20.4
13193,469.72,24.8


Obviously, these are invalid latitude and/or longitude values.

#### do_date_check, do_time_check

Now, we focus on the datetime. Firsly, we have to convert the strings representing datetimes to real datetime objects.

In [12]:
df["messzeit_dt"] = pd.to_datetime(df["messzeit"])
df["messzeit_dt"]

0       2026-04-07 01:00:00
1       2026-04-07 03:00:00
2       2026-04-07 05:00:00
3       2026-04-07 07:00:00
4       2026-04-07 09:00:00
                ...        
18166   2026-04-07 19:00:00
18167   2026-04-07 20:00:00
18168   2026-04-07 21:00:00
18169   2026-04-07 22:00:00
18170   2026-04-07 23:00:00
Name: messzeit_dt, Length: 18171, dtype: datetime64[ns]

In [13]:
date_qc = do_date_check(
    date=df["messzeit_dt"],
    year_init=2000,
    year_end=2030,
)
date_qc

0        0
1        0
2        0
3        0
4        0
        ..
18166    0
18167    0
18168    0
18169    0
18170    0
Length: 18171, dtype: int64

In [14]:
df[date_qc == 1]["messzeit_dt"]

Series([], Name: messzeit_dt, dtype: datetime64[ns])

The date check shows that all dates are valid. This is not suprising since an equivalent tests was done before providing the data. 
Reports with invalid dates are deselected.

Now, let's have a look at the times.

In [15]:
time_qc = do_time_check(
    date=df["messzeit_dt"],
)
time_qc

0        0
1        0
2        0
3        0
4        0
        ..
18166    0
18167    0
18168    0
18169    0
18170    0
Length: 18171, dtype: int64

In [16]:
df[time_qc == 1]["messzeit_dt"]

Series([], Name: messzeit_dt, dtype: datetime64[ns])

As expected, the result is equal to the date check.

#### do_missing_value_check

A pre-processing step is to convert the the seas surface temperature from `°C` to `K` .

In [17]:
df["wassertemperatur_kelvin"] = df["wassertemperatur"] + 273.15
df["wassertemperatur_kelvin"]

0       NaN
1       NaN
2       NaN
3       NaN
4       NaN
         ..
18166   NaN
18167   NaN
18168   NaN
18169   NaN
18170   NaN
Name: wassertemperatur_kelvin, Length: 18171, dtype: float64

Now, let's flag all missing observed values as failed.

In [18]:
miss_qc = do_missing_value_check(
    df["wassertemperatur_kelvin"]
)
miss_qc

0        1
1        1
2        1
3        1
4        1
        ..
18166    1
18167    1
18168    1
18169    1
18170    1
Length: 18171, dtype: int64

In [19]:
df[miss_qc == 1]["wassertemperatur_kelvin"]

0       NaN
1       NaN
2       NaN
3       NaN
4       NaN
         ..
18166   NaN
18167   NaN
18168   NaN
18169   NaN
18170   NaN
Name: wassertemperatur_kelvin, Length: 13475, dtype: float64

In [20]:
miss_qc.value_counts()

1    13475
0     4696
Name: count, dtype: int64

The main part is missing data.

#### do_hard_limit_check

Let's flag sea surface temperatures less than `-4.0 °C` and more than `45.0 °C` as failed.

In [21]:
limit_qc = do_hard_limit_check(
    df["wassertemperatur_kelvin"],
    limits=[-4.0 + 273.15, 45.0 + 273.15],
)
limit_qc

0        2
1        2
2        2
3        2
4        2
        ..
18166    2
18167    2
18168    2
18169    2
18170    2
Length: 18171, dtype: int64

In [22]:
df[limit_qc == 1]["wassertemperatur_kelvin"]

323    320.15
Name: wassertemperatur_kelvin, dtype: float64

We get expected results again.

#### do_sst_freeze_check

Let's flag observed sea surface temperatures below the freezing point of sea water (`-1.8 °C`) as failed.

In [23]:
freeze_qc = do_sst_freeze_check(
    df["wassertemperatur_kelvin"],
    freezing_point=-1.8 + 273.15,
)
freeze_qc

0        2
1        2
2        2
3        2
4        2
        ..
18166    2
18167    2
18168    2
18169    2
18170    2
Length: 18171, dtype: int64

In [24]:
df[freeze_qc == 1]["wassertemperatur_kelvin"]

Series([], Name: wassertemperatur_kelvin, dtype: float64)

#### do_multiple_individual_check

We can apply `do_hard_limit_check` and `do_sst_freeze_check` with one call using `do_multiple_individual_check`.

Therfor, we need a Quality Control dictionary containig the relevant information.

In [25]:
qc_dict = {
    "HARD": {
        "func": "do_hard_limit_check",
        "names": {"value": "wassertemperatur_kelvin"},
        "arguments": {"limits": [-4.0 + 273.15, 45.0 + 273.15]},
    },
    "FREEZE": {
        "func": "do_sst_freeze_check",
        "names": {"sst": "wassertemperatur_kelvin"},
        "arguments": {"freezing_point": -1.8 + 273.15},
    }
}

Let's do the checks. As soon as a test fails, the subsequent ones are no longer run (`return_method='failed'`).

In [26]:
df_sst_qc = do_multiple_individual_check(
    df,
    qc_dict,
    return_method="failed",
)
df_sst_qc

,HARD,FREEZE
0,2,2
1,2,2
2,2,2
3,2,2
4,2,2
...,...,...
18166,2,2
18167,2,2
18168,2,2
18169,2,2


We got a DataFrame containing QC flags for each test:

* `0`: passed
* `1`: failed
* `2`: not checked (this is for missing values)

We write a little helper function to get one over-all QC flag.

In [27]:
import numpy as np

def get_single_qc_flag(df):
    """Get single QC flag from DataFrame containing multiple QC flags."""
    mask_0 = (df == 0).any(axis=1)
    mask_1 = (df == 1).any(axis=1)
    mask_2 = (df == 2).any(axis=1)
    mask_3 = (df == 3).any(axis=1)

    conditions = [mask_1, mask_0, mask_3, mask_2]
    choices = [1, 0, 3, 2]
    result = np.select(conditions, choices, default=2)
    return pd.Series(result, index=df.index, name="QC_FLAG")

In [28]:
sst_qc = get_single_qc_flag(df_sst_qc)
sst_qc

0        2
1        2
2        2
3        2
4        2
        ..
18166    2
18167    2
18168    2
18169    2
18170    2
Name: QC_FLAG, Length: 18171, dtype: int64

In [29]:
df[sst_qc == 1]["wassertemperatur_kelvin"]

323    320.15
Name: wassertemperatur_kelvin, dtype: float64

In [30]:
df[sst_qc == 2]["wassertemperatur_kelvin"]

0       NaN
1       NaN
2       NaN
3       NaN
4       NaN
         ..
18166   NaN
18167   NaN
18168   NaN
18169   NaN
18170   NaN
Name: wassertemperatur_kelvin, Length: 13475, dtype: float64

The results are as expected.

#### do_supersaturation_check, do_wind_consistency_check

Finally, we do soem checks that compare two observed variables.

We take air temperature and dew point temperature to tests supersaturation. We convert both of them from `°C` to `K`.

In [31]:
df["lufttemperatur_kelvin"] = df["lufttemperatur"] + 273.15 
df["lufttemperatur_kelvin"]

0           NaN
1           NaN
2           NaN
3           NaN
4           NaN
          ...  
18166    298.75
18167    298.65
18168    298.45
18169    298.45
18170    298.25
Name: lufttemperatur_kelvin, Length: 18171, dtype: float64

In [32]:
df["taupunkttemperatur_kelvin"] = df["taupunkttemperatur"] + 273.15
df["taupunkttemperatur_kelvin"]

0           NaN
1           NaN
2           NaN
3           NaN
4           NaN
          ...  
18166    295.45
18167    295.95
18168    295.95
18169    295.65
18170    294.75
Name: taupunkttemperatur_kelvin, Length: 18171, dtype: float64

In [33]:
super_qc = do_supersaturation_check(
    at2 = df["lufttemperatur_kelvin"],
    dpt = df["taupunkttemperatur_kelvin"],
)
super_qc

0        2
1        2
2        2
3        2
4        2
        ..
18166    0
18167    0
18168    0
18169    0
18170    0
Length: 18171, dtype: int64

In [34]:
df[super_qc == 1][["lufttemperatur_kelvin", "taupunkttemperatur_kelvin"]]

,lufttemperatur_kelvin,taupunkttemperatur_kelvin
1229,291.15,294.75
1230,289.15,294.55
17147,299.25,300.85
17148,299.15,300.75
17149,299.15,300.75
17150,299.35,300.95
17151,300.05,301.75
17152,299.95,301.65
17153,300.05,301.65
17154,299.95,301.65


Supersaturation is when dew point temperature is higher than air temperature.

The last check is a wind consistency check. 

Zero windspeed should correspond to no particular direction (variable) and wind speeds above a threshold should correspond to a particular direction.

In [35]:
wind_qc = do_wind_consistency_check(
    wind_speed = df["windgeschwindigkeit"],
    wind_direction = df["windrichtung"]
)
wind_qc

0        2
1        2
2        2
3        2
4        2
        ..
18166    0
18167    0
18168    0
18169    0
18170    0
Length: 18171, dtype: int64

In [36]:
df[wind_qc == 1][["windgeschwindigkeit", "windrichtung"]]

,windgeschwindigkeit,windrichtung
1089,0.0,360.0
1092,0.0,360.0
1093,0.0,360.0
4669,0.0,110.0
4670,0.0,110.0
4674,0.0,250.0
4977,0.0,350.0
12478,0.0,10.0
12479,0.0,10.0
12667,0.0,340.0


If wind speed is `0`, wind direction should be `0` too.

### Map data to the CDM

Now, we map the MAROB data to the CDM.

In [37]:
db_marob = map_model(df, imodel="marob")

2026-04-08 15:02:54,725 - root - INFO - Initialized basic logging configuration successfully


In [38]:
db_marob

header                                     \
                     report_id region sub_region application_area   
0      DWD_MAROBSHIP-441917752   <NA>       <NA>   [1, 7, 10, 11]   
1      DWD_MAROBSHIP-441936176   <NA>       <NA>   [1, 7, 10, 11]   
2      DWD_MAROBSHIP-441955660   <NA>       <NA>   [1, 7, 10, 11]   
3      DWD_MAROBSHIP-441973355   <NA>       <NA>   [1, 7, 10, 11]   
4      DWD_MAROBSHIP-441991207   <NA>       <NA>   [1, 7, 10, 11]   
...                        ...    ...        ...              ...   
18166  DWD_MAROBSHIP-442063883   <NA>       <NA>   [1, 7, 10, 11]   
18167  DWD_MAROBSHIP-442072460   <NA>       <NA>   [1, 7, 10, 11]   
18168  DWD_MAROBSHIP-442081846   <NA>       <NA>   [1, 7, 10, 11]   
18169  DWD_MAROBSHIP-442090249   <NA>       <NA>   [1, 7, 10, 11]   
18170  DWD_MAROBSHIP-442099690   <NA>       <NA>   [1, 7, 10, 11]   

                                                                               \
      observing_programme report_type station_name station_type platform_type   
0                    [56]           0         <NA>            2             2   
1                    [56]           0         <NA>            2             2   
2                    [56]           0         <NA>            2             2   
3                    [56]           0         <NA>            2             2   
4                    [56]           0         <NA>            2             2   
...                   ...         ...          ...          ...           ...   
18166                [56]           0         <NA>            2             2   
18167                [56]           0         <NA>            2             2   
18168                [56]           0         <NA>            2             2   
18169                [56]           0         <NA>            2             2   
18170                [56]           0         <NA>            2             2   

                         ... observations-slp                    \
      platform_sub_type  ...   original_value conversion_method   
0                  <NA>  ...           1008.0                 7   
1                  <NA>  ...           1007.0                 7   
2                  <NA>  ...           1007.0                 7   
3                  <NA>  ...           1019.4                 7   
4                  <NA>  ...           1019.2                 7   
...                 ...  ...              ...               ...   
18166              <NA>  ...             <NA>              <NA>   
18167              <NA>  ...             <NA>              <NA>   
18168              <NA>  ...             <NA>              <NA>   
18169              <NA>  ...             <NA>              <NA>   
18170              <NA>  ...             <NA>              <NA>   

                                                                               \
      processing_code processing_level adjustment_id traceability advanced_qc   
0                <NA>                3          <NA>            2           0   
1                <NA>                3          <NA>            2           0   
2                <NA>                3          <NA>            2           0   
3                <NA>                3          <NA>            2           0   
4                <NA>                3          <NA>            2           0   
...               ...              ...           ...          ...         ...   
18166             NaN             <NA>           NaN         <NA>        <NA>   
18167             NaN             <NA>           NaN         <NA>        <NA>   
18168             NaN             <NA>           NaN         <NA>        <NA>   
18169             NaN             <NA>           NaN         <NA>        <NA>   
18170             NaN             <NA>           NaN         <NA>        <NA>   

                                                              
      advanced_uncertainty advanced_homogenisation source_id  
0                   

In [39]:
db_marob["header"]

,report_id,region,sub_region,application_area,observing_programme,report_type,station_name,station_type,platform_type,platform_sub_type,...,events_at_station,report_quality,duplicate_status,duplicates,record_timestamp,history,processing_level,processing_codes,source_id,source_record_id
0,DWD_MAROBSHIP-441917752,<NA>,<NA>,"[1, 7, 10, 11]",[56],0,<NA>,2,2,<NA>,...,<NA>,2,4,<NA>,2026-04-08 13:02:56.973711+00:00,2026-04-08 13:02:56. Initial conversion from D...,<NA>,<NA>,10384,441917752
1,DWD_MAROBSHIP-441936176,<NA>,<NA>,"[1, 7, 10, 11]",[56],0,<NA>,2,2,<NA>,...,<NA>,2,4,<NA>,2026-04-08 13:02:56.973711+00:00,2026-04-08 13:02:56. Initial conversion from D...,<NA>,<NA>,10384,441936176
2,DWD_MAROBSHIP-441955660,<NA>,<NA>,"[1, 7, 10, 11]",[56],0,<NA>,2,2,<NA>,...,<NA>,2,4,<NA>,2026-04-08 13:02:56.973711+00:00,2026-04-08 13:02:56. Initial conversion from D...,<NA>,<NA>,10384,441955660
3,DWD_MAROBSHIP-441973355,<NA>,<NA>,"[1, 7, 10, 11]",[56],0,<NA>,2,2,<NA>,...,<NA>,2,4,<NA>,2026-04-08 13:02:56.973711+00:00,2026-04-08 13:02:56. Initial conversion from D...,<NA>,<NA>,10384,441973355
4,DWD_MAROBSHIP-441991207,<NA>,<NA>,"[1, 7, 10, 11]",[56],0,<NA>,2,2,<NA>,...,<NA>,2,4,<NA>,2026-04-08 13:02:56.973711+00:00,2026-04-08 13:02:56. Initial conversion from D...,<NA>,<NA>,10384,441991207
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18166,DWD_MAROBSHIP-442063883,<NA>,<NA>,"[1, 7, 10, 11]",[56],0,<NA>,2,2,<NA>,...,<NA>,2,4,<NA>,2026-04-08 13:02:56.973711+00:00,2026-04-08 13:02:56. Initial conversion from D...,<NA>,<NA>,10384,442063883
18167,DWD_MAROBSHIP-442072460,<NA>,<NA>,"[1, 7, 10, 11]",[56],0,<NA>,2,2,<NA>,...,<NA>,2,4,<NA>,2026-04-08 13:02:56.973711+00:00,2026-04-08 13:02:56. Initial conversion from D...,<NA>,<NA>,10384,442072460
18168,DWD_MAROBSHIP-442081846,<NA>,<NA>,"[1, 7, 10, 11]",[56],0,<NA>,2,2,<NA>,...,<NA>,2,4,<NA>,2026-04-08 13:02:56.973711+00:00,2026-04-08 13:02:56. Initial conversion from D...,<NA>,<NA>,10384,442081846
18169,DWD_MAROBSHIP-442090249,<NA>,<NA>,"[1, 7, 10, 11]",[56],0,<NA>,2,2,<NA>,...,<NA>,2,4,<NA>,2026-04-08 13:02:56.973711+00:00,2026-04-08 13:02:56. Initial conversion from D...,<NA>,<NA>,10384,442090249
